In [1]:
import pandas as pd

In [2]:
ledger = pd.read_csv('../01_data/raw/ledger.csv')
gateway = pd.read_csv('../01_data/raw/gateway.csv')

print("Ledger Columns:", ledger.columns.tolist())
print("Gateway Columns:", gateway.columns.tolist())

Ledger Columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']
Gateway Columns: ['transaction_id', 'transaction_date', 'merchant_id', 'amount_usd', 'status', 'payment_method']


In [4]:
# Check for exact duplicate rows
print("Ledger duplicated()", ledger.duplicated())
print("Ledger Duplicates:", ledger.duplicated().sum())
print("Gateway Duplicates:", gateway.duplicated().sum())

# Check for missing (null) values in each column
print("\n--- Ledger Nulls ---")
print(ledger.isnull().sum())

print("\n--- Gateway Nulls ---")
print(gateway.isnull().sum())


Ledger duplicated() 0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
dtype: bool
Ledger Duplicates: 0
Gateway Duplicates: 0

--- Ledger Nulls ---
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64

--- Gateway Nulls ---
transaction_id      0
transaction_date    0
merchant_id         0
amount_usd          0
status              0
payment_method      0
dtype: int64


In [8]:
# Merge the two tables on transaction_id
recon = pd.merge(ledger, gateway, on='transaction_id', how='outer', suffixes=('_ledger', '_gateway'), indicator=True)
print(recon)
# Find records missing in Gateway
missing_in_gateway = recon[recon['_merge'] == 'left_only']
print("Missing in Gateway count:", len(missing_in_gateway))
missing_in_gateway.to_csv('../01_data/processed/missing_in_gateway.csv', index=False)

# Find records missing in Ledger
missing_in_ledger = recon[recon['_merge'] == 'right_only']
print("Missing in Ledger count:", len(missing_in_ledger))
missing_in_ledger.to_csv('../01_data/processed/missing_in_ledger.csv', index=False)


   transaction_id transaction_date_ledger merchant_id_ledger  \
0            R001              2026-03-01               M001   
1            R002              2026-03-01               M002   
2            R003              2026-03-02               M001   
3            R004              2026-03-02               M003   
4            R005              2026-03-03               M004   
5            R006              2026-03-03               M002   
6            R007              2026-03-04               M005   
7            R008              2026-03-04               M001   
8            R009              2026-03-05               M002   
9            R010              2026-03-05               M004   
10           R011                     NaN                NaN   

    amount_usd_ledger status_ledger payment_method_ledger  \
0              1200.0       success                   UPI   
1               850.0       success                  Card   
2               500.0       success             

In [10]:
# First, filter the data to only look at records that exist in BOTH systems
matched_records = recon[recon['_merge'] == 'both'].copy()

# Find Amount Mismatches
amount_mismatches = matched_records[matched_records['amount_usd_ledger'] != matched_records['amount_usd_gateway']]
print("Amount Mismatches count:", len(amount_mismatches))
amount_mismatches.to_csv('../01_data/processed/amount_mismatches.csv', index=False)

# Find Status Mismatches
status_mismatches = matched_records[matched_records['status_ledger'] != matched_records['status_gateway']]
print("Status Mismatches count:", len(status_mismatches))
status_mismatches.to_csv('../01_data/processed/status_mismatches.csv', index=False)


Amount Mismatches count: 2
Status Mismatches count: 1


In [11]:
import json

# 1. Build the final Reconciliation Report
def determine_issue(row):
    if row['_merge'] == 'left_only':
        return 'Missing in Gateway'
    elif row['_merge'] == 'right_only':
        return 'Missing in Ledger'
    elif row['amount_usd_ledger'] != row['amount_usd_gateway']:
        return 'Amount Mismatch'
    elif row['status_ledger'] != row['status_gateway']:
        return 'Status Mismatch'
    else:
        return 'Matched'

recon['reconciliation_issue'] = recon.apply(determine_issue, axis=1)
recon.to_csv('../01_data/processed/reconciliation_report.csv', index=False)
print("Reconciliation report saved!")

# 2. Generate Summary Metrics JSON
issues_df = recon[recon['reconciliation_issue'] != 'Matched']

metrics = {
  "total_ledger_rows": len(ledger),
  "total_gateway_rows": len(gateway),
  "missing_in_gateway_count": len(missing_in_gateway),
  "missing_in_ledger_count": len(missing_in_ledger),
  "amount_mismatch_count": len(amount_mismatches),
  "status_mismatch_count": len(status_mismatches),
  "reconciliation_issue_count": len(issues_df),
  "ledger_total_amount": float(ledger['amount_usd'].sum()),
  "gateway_total_amount": float(gateway['amount_usd'].sum()),
  "amount_at_risk": float(issues_df['amount_usd_ledger'].fillna(issues_df['amount_usd_gateway']).sum())
}

with open('summary_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=4)
    
print("Summary metrics saved successfully! Here is your output:")
print(json.dumps(metrics, indent=2))


Reconciliation report saved!
Summary metrics saved successfully! Here is your output:
{
  "total_ledger_rows": 10,
  "total_gateway_rows": 9,
  "missing_in_gateway_count": 2,
  "missing_in_ledger_count": 1,
  "amount_mismatch_count": 2,
  "status_mismatch_count": 1,
  "reconciliation_issue_count": 6,
  "ledger_total_amount": 23340.0,
  "gateway_total_amount": 20550.0,
  "amount_at_risk": 15090.0
}


In [12]:
import json

# 1. Load the JSON file
with open('../01_data/raw/api_response_sample.json', 'r') as f:
    api_data = json.load(f)

# 2. Flatten the nested JSON
# We extract the 'settlements' list, and attach the higher-level batch and merchant info to each row
normalized_df = pd.json_normalize(
    api_data['batches'], 
    record_path='settlements', 
    meta=[
        'batch_id', 
        ['merchant', 'merchant_id'], 
        ['merchant', 'merchant_name'], 
        ['merchant', 'region']
    ]
)

# 3. Clean column names (Pandas uses '.' for nested dicts, we will change them to '_')
normalized_df.columns = [col.replace('.', '_') for col in normalized_df.columns]

# 4. Convert date/time fields from text to proper DateTime objects
normalized_df['processed_at'] = pd.to_datetime(normalized_df['processed_at'])

# 5. Save the normalized output
normalized_df.to_csv('../01_data/processed/api_normalized.csv', index=False)

print("JSON successfully flattened and saved! Here is a preview:")
display(normalized_df)


JSON successfully flattened and saved! Here is a preview:


,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,batch_id,merchant_merchant_id,merchant_merchant_name,merchant_region
0,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN,B001,M001,Alpha Mart,APAC
1,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN,B001,M001,Alpha Mart,APAC
2,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG,B001,M001,Alpha Mart,APAC
3,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US,B002,M004,Delta Travels,US
4,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US,B002,M004,Delta Travels,US
5,S006,7200.0,settled,2026-03-07 09:30:00+00:00,Bank C,US,B002,M004,Delta Travels,US
